# 1. 5-SEED PAIRED ALIGNMENT COMPARISON — OLD vs CORRECTED, Condition D — AeroSync-Mamba v24

**Why.** v23 built two preprocessing paths:
- **OLD_ALIGNMENT**: audio anchored at t = 0, 99 missions, identical to v18.
- **CORRECTED_ALIGNMENT**: audio anchored at `recording_started_time`, 4 missions with offset/duration ≥ 0.5 excluded,
  95 missions, windows start at the audio onset.

On seed 42 with Condition D (no HTT time embedding, no LACA lag bias), macro-F1 barely moved (−0.0050, 0.31× the
0.0160 noise floor). But the aggregate hid a large swing: **Class 4 recall −0.1238** and **Class 3 recall +0.0693**.
v24 tests whether that swing replicates at 5 seeds.

**Design.**
- 5 seeds (42, 123, 2024, 7, 99) × 2 alignments = **10 runs**, run in pairs: OLD then CORRECTED for each seed.
- Both alignments of a seed start from the same initial weights and RNG stream. Every condition shares v18's per-seed
  `init_sha1` / `rng_sha1`, which the notebook checks.
- v23's data cells run unchanged. The code hashes and data fingerprints are checked against v18 (OLD) and v23 (CORRECTED).

**Seed-42 reuse.** v23's seed-42 runs are reused **only if their saved test probabilities are available**. Attach
v23's output (`v23_test_probs/`) as a Kaggle input. The probabilities must reproduce v23's logged metrics and
confusion matrices exactly, and the data and initial weights must match. Otherwise seed 42 is **retrained** and checked
against v23's numbers (±0.0005). The per-window controls below need the per-window probabilities, and v23's CSV has
only the aggregates, so a reuse without probabilities is not possible. That gives 8 runs if reused, 10 if not.

**Controls on each seed's OLD model** (same weights, only the scored windows change):
1. **Mission control (requested; replicates v23).** OLD scored on CORRECTED's test missions. In v23 the only test
   mission CORRECTED drops is a **class-2** mission. So this control leaves the Class 3/4 windows untouched **by
   construction**, and it cannot show a Class 3/4 swing at any seed. The notebook prints which classes the dropped
   missions belong to.
2. **Time-matched control (added; no training).** OLD scored only on windows of CORRECTED's test missions that start at
   or after that mission's audio onset. This is the same flight phase that CORRECTED scores. The Class 3/4 missions are
   identical between the two test sets. What differs is that CORRECTED drops the first ~19 s of every flight, so this
   is the control that separates **"which part of the flight is scored"** from **"audio correctly aligned"**.

**Class-level training-data diagnostic.** Training windows per class, OLD → CORRECTED, with the % retained. This checks
whether Class 4 lost disproportionately many training examples.

**Statistics.** For macro-F1, Class 3 recall and Class 4 recall, the per-seed paired difference CORRECTED − OLD is
reported as mean, SD, wins/5 and a paired t-test. The **paired t-test is primary**: with n = 5, the exact two-sided
Wilcoxon signed-rank p cannot go below 2/2⁵ = 0.0625, so it can never reach 0.05. Wilcoxon is printed as a robustness
check. The test sets differ in composition between the two alignments, so these are paired by seed, not by window.

Outputs: `alignment_5seed_paired_results.csv` (written after every run), `v24_checkpoints/`, `v24_test_probs/`.


## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import inspect
import json
import math
import os
import random
import subprocess
import sys
import textwrap
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table in v22)
# ==============================================================================
# --- v24: 5 seeds x ONE architecture (v22 Condition D) x TWO datasets, paired by seed ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as every prior multi-seed experiment
CONDITION = "D"                          # v22 Condition D = current best configuration
HTT_TIME_EMBED_ENABLED = False           # HTT returns norm(z + e_m): no time embedding
LACA_LAG_BIAS_ENABLED = False            # LACA beta exactly 0.0: no lag bias
ALIGNMENTS = ["OLD_ALIGNMENT", "CORRECTED_ALIGNMENT"]   # per seed: OLD then CORRECTED
RUN_ORDER = [(seed, a) for seed in SEEDS for a in ALIGNMENTS]   # 5 back-to-back pairs = 10 slots
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                          # beta if LACA_LAG_BIAS_ENABLED were True (it is not: Condition D -> 0.0)
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params (built, unused)
LAG_BIAS_PARAMS = 97
LAG_BIAS_LN_EPS = 1e-5
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- v23: corrected temporal anchor (CORRECTED_ALIGNMENT only — OLD_ALIGNMENT reads none of this) ---
# audio_start_offset_s = (recording_started_time - sensor_t0) / 1e6, both raw microsecond timestamps of the same clock
AUDIO_EXCLUDE_RATIO = 0.5                # exclude a mission if audio_start_offset_s / mission_duration_s >= this
EXPECTED_EXCLUDED_MISSIONS = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                              "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = {"OLD_ALIGNMENT": 99, "CORRECTED_ALIGNMENT": 95}
KNOWN_AUDIO_OFFSETS_S = {"rosbag2_2021_12_22-09_14_31": 24.38}   # earlier investigation (motor-start event at t = 24.5 s)
KNOWN_OFFSET_TOLERANCE_S = 0.05
EXPECTED_MEDIAN_OFFSET_S = 19.2          # median over all missions, earlier investigation
MEDIAN_OFFSET_TOLERANCE_S = 1.0
N_OFFSETS_TO_PRINT = 8
# Where CORRECTED windows start: "audio_onset" = the 2.56 s / 1.28 s grid starts at the mission's audio onset, so every
# window lies where BOTH modalities were recorded (v18 already ends the grid at min(sensor end, audio end)).
# "sensor_t0" = v18's grid start (t = 0); windows before the onset then get all-zero audio (np.interp left=0.0).
CORRECTED_WINDOW_START = "audio_onset"
# Strongest-64 bins for CORRECTED: "own_train" = recomputed (same method) from CORRECTED's own training missions;
# "old_bins" = reuse OLD_ALIGNMENT's frozen bins.
CORRECTED_BIN_SOURCE = "own_train"
assert CORRECTED_WINDOW_START in ("audio_onset", "sensor_t0") and CORRECTED_BIN_SOURCE in ("own_train", "old_bins")

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v24_checkpoints"
PROBS_DIR = WORK_DIR / "v24_test_probs"
RESULTS_CSV = WORK_DIR / "alignment_5seed_paired_results.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Reference numbers ---
# v22 Condition D (HTT-time OFF, LACA-lag OFF), seed 42 — Artifacts/Version 22/temporal_2x2_ablation_results.csv.
# Same data, architecture and seed as OLD_ALIGNMENT here: THIS is what OLD_ALIGNMENT must reproduce.
V22_D_SEED42 = {"accuracy": 0.9097345132743363, "macro_f1": 0.90640272771588, "balanced_accuracy": 0.9040236181473512,
                "macro_auroc": 0.9890678208140686, "class3_recall": 0.8859649122807017, "class4_recall": 0.8706467661691543,
                "confusion_matrix": [[284, 0, 0, 0, 2], [3, 197, 4, 0, 1], [0, 1, 170, 38, 1], [0, 0, 10, 202, 16], [0, 0, 3, 23, 175]],
                "best_epoch": 17, "init_sha1": "5850841e699f", "rng_sha1": "5641b401720d"}
# v18 beta=0.0, seed 42 = v22 Condition A (HTT time embedding ON). A DIFFERENT architecture from Condition D:
# printed for context only, NOT a reproduction target.
V18_BETA0_SEED42 = {"accuracy": 0.9088495575221239, "macro_f1": 0.9057330017972403, "balanced_accuracy": 0.9030033658152373,
                    "macro_auroc": 0.9891867537737307, "class3_recall": 0.8947368421052632, "class4_recall": 0.8507462686567164}
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1 = the noise floor here
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v22)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}
# sha1[:16] of the source of every v18 data function / method (textwrap.dedent + rstrip). The two renamed functions are
# hashed under their v18 names. OLD_ALIGNMENT's code is checked against these: identical hash = identical code.
V18_DATA_SOURCE_SHA1 = {
    "parse_mission": "48132d5e84609fc4",
    "split_missions": "e60744ab660df986",
    "select_strongest_bins": "fb7d15f798514139",
    "audio_features": "d957dda051fb003d",
    "slice_windows": "0d493fd15803e0de",
    "UAVWindowDataset.__init__": "4a7708aec1baca55",
    "UAVWindowDataset.__len__": "15a9aa16ef54ea0a",
    "UAVWindowDataset.__getitem__": "5b6e2898c53b5989",
    "split_fingerprint": "195d93e4771675ca",
}

# --- v24 references ---
# Initial weights / torch RNG right after construction, per seed (v18 CSV; identical for every v22 condition and for
# both v23 alignments). Both alignments of a seed must start here.
SEED_INIT_REFERENCE = {42: ("5850841e699f", "5641b401720d"), 123: ("74be0883c62b", "0a4b3be44b18"),
                       2024: ("92852955eef1", "619c0e0a6b4d"), 7: ("7f5d0cd2e7cb", "9fdcf777d285"),
                       99: ("e40c78169f57", "81dcd4638a88")}
# v23 seed-42 results (Artifacts/Version 23/alignment_sanity_v23_results.csv) — the reuse / reproduction target.
V23_SEED42 = {
    "OLD_ALIGNMENT": {"accuracy": 0.9097345132743363, "macro_f1": 0.90640272771588, "balanced_accuracy": 0.9040236181473512,
                      "macro_auroc": 0.9890678208140686, "class3_recall": 0.8859649122807017, "class4_recall": 0.8706467661691543,
                      "confusion_matrix": [[284, 0, 0, 0, 2], [3, 197, 4, 0, 1], [0, 1, 170, 38, 1], [0, 0, 10, 202, 16], [0, 0, 3, 23, 175]],
                      "best_epoch": 17, "best_val_macro_f1": 0.908769, "n_test_windows": 1130},
    "CORRECTED_ALIGNMENT": {"accuracy": 0.9079754601226994, "macro_f1": 0.9014428634096197, "balanced_accuracy": 0.8939724876561621,
                            "macro_auroc": 0.992763216249706, "class3_recall": 0.9553072625698324, "class4_recall": 0.7468354430379747,
                            "confusion_matrix": [[212, 2, 0, 0, 0], [0, 153, 2, 1, 0], [0, 2, 86, 19, 1], [0, 0, 3, 171, 5], [0, 1, 0, 39, 118]],
                            "best_epoch": 15, "best_val_macro_f1": 0.927039, "n_test_windows": 815},
    "OLD_ON_CORRECTED_MISSIONS": {"macro_f1": 0.9040815922533835, "class3_recall": 0.8859649122807017,
                                  "class4_recall": 0.8706467661691543, "n_test_windows": 1053},
}
# Seed-42 reuse: v23's saved test probabilities/labels (attach v23's output as a Kaggle input). "auto" = reuse when the
# files are found AND reproduce V23_SEED42 exactly AND the data + initial weights match; otherwise retrain seed 42.
REUSE_SEED42 = "auto"                    # "auto" | "never"
V23_PROBS_GLOBS = ["/kaggle/input/**/v23_test_probs/{name}", str(WORK_DIR / "v23_test_probs" / "{name}")]
# CORRECTED_ALIGNMENT as v23 built it (v23 run output) + v23's CORRECTED function sources (sha1, same scheme as v18's).
V23_CORRECTED_DATA = {"n_missions": 95, "missions": {"train": 67, "val": 13, "test": 15},
                      "windows": {"train": 3737, "val": 686, "test": 815},
                      "class_counts": {"train": (701, 765, 706, 796, 769), "test": (214, 156, 108, 179, 158)}}
V23_CORRECTED_SOURCE_SHA1 = {"parse_mission_CORRECTED_ALIGNMENT": "49d4b2f5a1359025", "slice_windows_CORRECTED_ALIGNMENT": "8cf47da958eda742"}
# Verdict rules.
PAIRED_ALPHA = 0.05
REPLICATION_MIN_SAME_SIGN = 4            # "replicates" = paired t p < alpha AND >= 4 of 5 seeds in seed 42's direction
DISPROPORTION_PP = 5.0                   # Class-4 retention this many points below the other classes' mean = disproportionate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"5-SEED PAIRED ALIGNMENT COMPARISON: Condition {CONDITION} (HTT-time {'ON' if HTT_TIME_EMBED_ENABLED else 'OFF'}, "
      f"LACA-lag {'ON' if LACA_LAG_BIAS_ENABLED else 'OFF'}) x seeds {SEEDS} x datasets {ALIGNMENTS} = {len(RUN_ORDER)} slots | "
      f"seed-42 reuse: {REUSE_SEED42} | results -> {RESULTS_CSV}")
print(f"CORRECTED_ALIGNMENT: audio anchored at recording_started_time | exclude offset/duration >= {AUDIO_EXCLUDE_RATIO} | "
      f"windows start at {CORRECTED_WINDOW_START} | bins {CORRECTED_BIN_SOURCE}")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA {LACA_NUM_HEADS} heads, beta 0.0 | Mamba "
      f"{MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | loss CE(class-weighted) + {LAMBDA_ALIGN} "
      f"InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")


Device: cuda (Tesla T4)
5-SEED PAIRED ALIGNMENT COMPARISON: Condition D (HTT-time OFF, LACA-lag OFF) x seeds [42, 123, 2024, 7, 99] x datasets ['OLD_ALIGNMENT', 'CORRECTED_ALIGNMENT'] = 10 slots | seed-42 reuse: auto | results -> /kaggle/working/alignment_5seed_paired_results.csv
CORRECTED_ALIGNMENT: audio anchored at recording_started_time | exclude offset/duration >= 0.5 | windows start at audio_onset | bins own_train
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA 4 heads, beta 0.0 | Mamba 2 x (d_state 16, d_conv 2) | loss CE(class-weighted) + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3a. Data — OLD_ALIGNMENT (v18's preprocessing, unchanged)
This is v18's data cell. Two functions are renamed and nothing else changes: `parse_mission` becomes
`parse_mission_OLD_ALIGNMENT`, and `slice_windows` becomes `slice_windows_OLD_ALIGNMENT`. All 99 missions are used, with
audio anchored at t = 0. At the end of the cell, every data function's source is hashed and compared with v18's, and
the bins and per-split fingerprints are compared with v18's reference values. Either a code change or a data change
would show up.

**v24 addition (bookkeeping only).** For each OLD window, the build loop now records its start time. It recomputes
the start with the same grid and skip rule that `slice_windows_OLD_ALIGNMENT` uses, and asserts that the recomputed
starts agree with every window's real timestamps. The time-matched control needs these starts. No function is changed,
and the v18 code and data checks below still apply.

In [3]:
# ==============================================================================
# DATA — OLD_ALIGNMENT: v18's preprocessing, UNCHANGED (audio anchored at t = 0, all 99 missions).
# parse missions -> mission-level split -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows
# with real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# v23: parse_mission -> parse_mission_OLD_ALIGNMENT and slice_windows -> slice_windows_OLD_ALIGNMENT (renamed only).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission_OLD_ALIGNMENT(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows_OLD_ALIGNMENT(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission_OLD_ALIGNMENT(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}
# v23: per-mission raw-data summary, so the CORRECTED parser can be checked to read the same samples and chunks.
OLD_MISSION_SUMMARY = {m.name: (len(p["sensor_times"]), len(p["audio_chunks"]), float(p["sensor_times"][-1]))
                       for m, p in parsed_missions.items() if p is not None}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
def _old_window_starts(mission):
    """v24 bookkeeping: the t_start of every window slice_windows_OLD_ALIGNMENT returns, in order (same grid from 0.0,
    same += STEP_SEC accumulation, same end rule, same sensor-count skip). Does not build any window."""
    s_times, a_times = mission["sensor_times"], mission["audio_times"]
    max_time = min(s_times[-1], a_times[-1])
    starts, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        if (((s_times >= t_start) & (s_times <= t_start + WINDOW_SEC)).sum()) >= MIN_SENSOR_SAMPLES_PER_WINDOW:
            starts.append(t_start)
        t_start += STEP_SEC
    return starts


def _starts_match_windows(mission, windows, starts):
    """v24 bookkeeping check: rebuild each window's tau_s exactly as slice_windows_OLD_ALIGNMENT does (the samples inside
    [t, t + WINDOW_SEC] in FILE order, evenly chosen by index) from the recomputed start t, and compare with the stored
    tau_s. Valid whether or not the IMU timestamps are sorted."""
    s_times = mission["sensor_times"]
    for w, t in zip(windows, starts):
        s_idx = np.where((s_times >= t) & (s_times <= t + WINDOW_SEC))[0]
        tau_s = (s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]] - t).astype(np.float32)
        if not np.array_equal(tau_s, w["tau_s"]):
            return False
    return True


split_windows, split_window_missions = {}, {}   # v23: the mission of every window is recorded next to it (same order)
split_window_starts = {}                        # v24: ... and its start time (s from the first IMU sample)
NON_MONOTONIC_SENSOR_MISSIONS = {}              # v24 diagnostic: missions whose IMU timestamps go backwards somewhere
for split in ("train", "val", "test"):
    split_windows[split], split_window_missions[split], split_window_starts[split] = [], [], []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                _w = slice_windows_OLD_ALIGNMENT(parsed_missions[m], label, SELECTED_BINS)
                split_windows[split].extend(_w)
                split_window_missions[split].extend([m.name] * len(_w))
                _st = _old_window_starts(parsed_missions[m])
                assert len(_st) == len(_w), f"{m.name}: {len(_st)} recomputed starts vs {len(_w)} windows"
                assert _starts_match_windows(parsed_missions[m], _w, _st), \
                    f"{m.name}: recomputed window starts disagree with the windows' own timestamps"
                _back = int((np.diff(parsed_missions[m]["sensor_times"]) < 0).sum())
                if _back:
                    NON_MONOTONIC_SENSOR_MISSIONS[m.name] = _back
                split_window_starts[split].extend(_st)
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}
print(f"[WINDOW STARTS] every OLD window's recomputed start verified by rebuilding its tau_s exactly ✅ | missions with "
      f"non-monotonic IMU timestamps: {len(NON_MONOTONIC_SENSOR_MISSIONS)}"
      + (f" {NON_MONOTONIC_SENSOR_MISSIONS} (backward steps per mission; v18's slicer picks samples in file order, so its "
         f"windows and these starts are unaffected — the rebuild above checks exactly that)" if NON_MONOTONIC_SENSOR_MISSIONS else ""))

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps (Condition D does not use them)")
del _tau_a, _tau_s

# --- 6. v23: OLD_ALIGNMENT is v18, byte for byte (code) and value for value (data) ---
def _source_sha1(obj, v18_name=None):
    """sha1[:16] of obj's source (dedented, rstripped); a renamed function is compared under its v18 name."""
    s = textwrap.dedent(inspect.getsource(obj)).rstrip()
    if v18_name is not None:
        s = s.replace(f"def {obj.__name__}(", f"def {v18_name}(", 1)
    return hashlib.sha1(s.encode()).hexdigest()[:16]


_old_code = {"parse_mission": (parse_mission_OLD_ALIGNMENT, "parse_mission"),
             "split_missions": (split_missions, None),
             "select_strongest_bins": (select_strongest_bins, None),
             "audio_features": (audio_features, None),
             "slice_windows": (slice_windows_OLD_ALIGNMENT, "slice_windows"),
             "UAVWindowDataset.__init__": (UAVWindowDataset.__init__, None),
             "UAVWindowDataset.__len__": (UAVWindowDataset.__len__, None),
             "UAVWindowDataset.__getitem__": (UAVWindowDataset.__getitem__, None),
             "split_fingerprint": (split_fingerprint, None)}
assert set(_old_code) == set(V18_DATA_SOURCE_SHA1)
OLD_SOURCE_CHECK = {}
for _name, (_obj, _v18_name) in _old_code.items():
    try:
        _h = _source_sha1(_obj, _v18_name)
        OLD_SOURCE_CHECK[_name] = _h == V18_DATA_SOURCE_SHA1[_name]
        print(f"[OLD == v18 CODE] {_name:<30} sha1 {_h} vs v18 {V18_DATA_SOURCE_SHA1[_name]} -> "
              + ("identical ✅" if OLD_SOURCE_CHECK[_name] else "🚨 DIFFERENT — v18's preprocessing was modified"))
    except (OSError, TypeError) as _e:
        OLD_SOURCE_CHECK[_name] = None
        print(f"[OLD == v18 CODE] {_name:<30} source not readable here ({_e}); the data fingerprint below still applies")
OLD_DATA_CHECK = {"strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
                  **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
                     REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")}}
for _name, _ok in OLD_DATA_CHECK.items():
    print(f"[OLD == v18 DATA] {_name:<58}: {'match ✅' if _ok else '🚨 MISMATCH'}")
OLD_N_MISSIONS = len(OLD_MISSION_SUMMARY)
print(f"[OLD MISSIONS] {OLD_N_MISSIONS} parsed missions (expected {EXPECTED_N_MISSIONS['OLD_ALIGNMENT']}) -> "
      + ("✅" if OLD_N_MISSIONS == EXPECTED_N_MISSIONS["OLD_ALIGNMENT"] else "🚨 UNEXPECTED COUNT"))
if not all(OLD_DATA_CHECK.values()) or False in OLD_SOURCE_CHECK.values():
    print("\n" + "🚨" * 40 + "\n🚨 OLD_ALIGNMENT does NOT reproduce v18's preprocessing — the OLD vs CORRECTED comparison "
          "is not against the historical pipeline.\n" + "🚨" * 40)
else:
    print("✅ OLD_ALIGNMENT = v18's preprocessing: identical code, identical strongest-64 bins, identical windows / sensor "
          "values / timestamps / class counts in every split. The 99-mission bin selection is v18's, unchanged.")

OLD_ALIGNMENT = {"name": "OLD_ALIGNMENT", "n_missions": OLD_N_MISSIONS, "bins": SELECTED_BINS,
                 "split_windows": split_windows, "window_missions": split_window_missions,
                 "window_starts": {s: np.array(v) for s, v in split_window_starts.items()},
                 "fingerprint": DATA_FINGERPRINT, "train_ds": train_ds, "val_ds": val_ds, "test_ds": test_ds,
                 "split_missions": {s: sorted({n for n in split_window_missions[s]}) for s in split_windows}}
del parsed_missions   # raw audio of 99 missions; the CORRECTED cell re-reads the raw files itself
gc.collect()


Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


Updating files: 100% (217/217), done.


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[WINDOW STARTS] every OLD window's recomputed start verified by rebuilding its tau_s exactly ✅ | missions with non-monotonic IMU timestamps: 99 {'rosbag2_2022_01_05-09_33_57': 5404, 'rosbag2_2022_01_05-09_29_43': 4898, 'rosbag2_2021_12_31-11_52_05': 5194, 'rosbag2_2021_12_31-14_57_10': 5351, 'rosbag2_2021_12_31-11_53_53': 5817, 'rosbag2_2022_01_05-10_23_31': 4515, 'rosbag2_2022_01_10-09_34_38': 4872, 'rosbag2_2021_12_31-13_2

28

## 3b. Data — CORRECTED_ALIGNMENT (new, alongside OLD)
- **Parser.** `parse_mission_CORRECTED_ALIGNMENT` is a copy of the OLD parser with one change: the audio time axis is
  anchored at `recording_started_time`. That value is read from this mission's first AudioBuffer record, never
  hardcoded. `sensor_t0` is the raw timestamp of the first IMU sample, the same one OLD subtracts.
- **Checks.**
  - The first few offsets are printed, and `rosbag2_2021_12_22-09_14_31` must read ≈ 24.38 s.
  - The median offset must be ≈ 19.2 s.
  - `recording_started_time` must be constant within each mission.
  - The sample and chunk counts must equal OLD's.
- **Exclusion.** Missions with `offset / duration ≥ 0.5` are excluded from **every** split of this dataset only. They
  are printed with their ratios and checked against the 4 expected names.
- **Split, bins, windows.** The split is the same as OLD's. Bins are recomputed from CORRECTED's training missions, and
  windows start at the audio onset (see the header).
- **Summary.** Both datasets' sizes are printed side by side at the end.

In [4]:
# ==============================================================================
# DATA — CORRECTED_ALIGNMENT: the SAME raw files and the SAME mission split, with the audio time axis anchored at the
# real recording start (recording_started_time) instead of t = 0. Missions whose audio starts in the second half of
# the flight are excluded from this dataset only. Nothing in the OLD_ALIGNMENT cell above is modified.
# ==============================================================================
def parse_mission_CORRECTED_ALIGNMENT(mission_dir: Path):
    """parse_mission_OLD_ALIGNMENT with ONE change — the audio time axis:
        OLD:       chunk_duration_s = mission_duration_s / n_chunks
                   audio_times      = arange(n_chunks) * chunk_duration_s
        CORRECTED: audio_start_offset_s = (recording_started_time - sensor_t0) / 1e6
                   chunk_duration_s     = (mission_duration_s - audio_start_offset_s) / n_chunks
                   audio_times          = audio_start_offset_s + arange(n_chunks) * chunk_duration_s
    recording_started_time: the FIRST AudioBuffer record's field (raw microseconds; constant within a mission).
    sensor_t0: raw microsecond timestamp of the first IMU sample (the sample OLD's sensor_times[0] refers to)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro, sensor_t0_us = [], [], [], None
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
                if sensor_t0_us is None:
                    sensor_t0_us = float(rec["timestamp"])
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks, recording_started_time, rst_values = [], None, set()
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                if "recording_started_time" in rec:
                    if recording_started_time is None:
                        recording_started_time = float(rec["recording_started_time"])
                    rst_values.add(float(rec["recording_started_time"]))
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    if recording_started_time is None:
        print(f"🚨 {mission_dir.name}: no recording_started_time in {audio_files[0].name} -> cannot anchor the audio; "
              f"mission left out of CORRECTED_ALIGNMENT")
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    mission_duration_s = float(sensor_times[-1] - sensor_times[0])
    audio_start_offset_s = (recording_started_time - sensor_t0_us) / 1e6
    chunk_duration_s = (mission_duration_s - audio_start_offset_s) / len(audio_chunks)
    audio_times = audio_start_offset_s + np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks,
            "recording_started_time": recording_started_time, "sensor_t0_us": sensor_t0_us,
            "n_distinct_recording_started_time": len(rst_values), "audio_start_offset_s": audio_start_offset_s,
            "mission_duration_s": mission_duration_s, "offset_ratio": audio_start_offset_s / mission_duration_s,
            "chunk_duration_s": chunk_duration_s, "old_chunk_duration_s": mission_duration_s / len(audio_chunks)}


def slice_windows_CORRECTED_ALIGNMENT(mission, label, bin_indices):
    """slice_windows_OLD_ALIGNMENT with ONE change — where the window grid starts:
    CORRECTED_WINDOW_START = "audio_onset": the first window starts at the audio onset (audio_times[0]), so every window
    lies inside the span where BOTH modalities were recorded (the end is min(sensor end, audio end), as in OLD).
    CORRECTED_WINDOW_START = "sensor_t0": OLD's start (t = 0); pre-onset windows then carry all-zero audio."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows = []
    t_start = max(0.0, float(a_times[0])) if CORRECTED_WINDOW_START == "audio_onset" else 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


# --- 1. parse every mission of the SAME split (mission_splits / split_dirs from the OLD cell: SPLIT_SEED 42) ---
corrected_parsed = {m: parse_mission_CORRECTED_ALIGNMENT(m) for s in split_dirs
                    for m in tqdm(split_dirs[s], desc=f"parse {s} (corrected)", leave=False)}
mission_home = {m: (s, label) for s in mission_splits for label in mission_splits[s] for m in mission_splits[s][label]}
_ok = {m: p for m, p in corrected_parsed.items() if p is not None}
_same_raw = [m.name for m, p in _ok.items()
             if OLD_MISSION_SUMMARY.get(m.name) != (len(p["sensor_times"]), len(p["audio_chunks"]), float(p["sensor_times"][-1]))]
print(f"[RAW DATA CHECK] {len(_ok)} missions parsed by the CORRECTED parser; IMU samples, audio chunks and duration equal "
      f"OLD's for " + ("every mission ✅" if not _same_raw and len(_ok) == len(OLD_MISSION_SUMMARY) else
                       f"all but {_same_raw} 🚨"))
_non_const = {m.name: p["n_distinct_recording_started_time"] for m, p in _ok.items() if p["n_distinct_recording_started_time"] != 1}
RST_CONSTANT = not _non_const
print("[METADATA] recording_started_time is constant within every mission ✅" if RST_CONSTANT else
      f"🚨 [METADATA] recording_started_time varies within {_non_const} (distinct values per mission) — the first "
      f"record's value is used, as specified")

# --- 2. verification: offsets read from the metadata ---
_by_name = {m.name: (m, p) for m, p in _ok.items()}
print(f"\n[OFFSETS] first {N_OFFSETS_TO_PRINT} missions (by name): audio_start_offset_s = (recording_started_time - "
      f"sensor_t0) / 1e6")
print(f"    {'mission':<30} {'class':>5} {'split':>5} {'sensor_t0 (us)':>18} {'rec_started (us)':>18} {'offset s':>9} "
      f"{'dur s':>8} {'ratio':>6} {'chunk ms OLD':>12} {'chunk ms NEW':>12}")
for _name in sorted(_by_name)[:N_OFFSETS_TO_PRINT] + [n for n in KNOWN_AUDIO_OFFSETS_S if n in _by_name]:
    _m, _p = _by_name[_name]
    _s, _lab = mission_home[_m]
    print(f"    {_name:<30} {_lab:>5} {_s:>5} {_p['sensor_t0_us']:>18.0f} {_p['recording_started_time']:>18.0f} "
          f"{_p['audio_start_offset_s']:>9.3f} {_p['mission_duration_s']:>8.2f} {_p['offset_ratio']:>6.3f} "
          f"{1e3 * _p['old_chunk_duration_s']:>12.3f} {1e3 * _p['chunk_duration_s']:>12.3f}")
KNOWN_OFFSET_CHECK = {}
for _name, _expected in KNOWN_AUDIO_OFFSETS_S.items():
    if _name not in _by_name:
        KNOWN_OFFSET_CHECK[_name] = False
        print(f"🚨 [KNOWN OFFSET] {_name} not found among the parsed missions")
        continue
    _got = _by_name[_name][1]["audio_start_offset_s"]
    KNOWN_OFFSET_CHECK[_name] = abs(_got - _expected) <= KNOWN_OFFSET_TOLERANCE_S
    print(f"[KNOWN OFFSET] {_name}: computed {_got:.3f} s vs earlier investigation ~{_expected:.2f} s -> "
          + ("✅ metadata read correctly" if KNOWN_OFFSET_CHECK[_name] else f"🚨 MISMATCH (> ±{KNOWN_OFFSET_TOLERANCE_S} s)"))
_offsets = np.array([p["audio_start_offset_s"] for p in _ok.values()])
MEDIAN_OFFSET_S = float(np.median(_offsets))
MEDIAN_OFFSET_OK = abs(MEDIAN_OFFSET_S - EXPECTED_MEDIAN_OFFSET_S) <= MEDIAN_OFFSET_TOLERANCE_S
print(f"[OFFSETS] over {len(_offsets)} missions: median {MEDIAN_OFFSET_S:.2f} s (expected ~{EXPECTED_MEDIAN_OFFSET_S}) "
      f"{'✅' if MEDIAN_OFFSET_OK else '🚨'} | min {_offsets.min():.2f} s | max {_offsets.max():.2f} s | "
      f"{int((_offsets < 0).sum())} negative")
print(f"[OLD MISPLACEMENT] under OLD_ALIGNMENT, chunk k sits at k*D/n but was recorded at offset + k*(D - offset)/n: the "
      f"audio is placed offset*(1 - k/n) too EARLY — {MEDIAN_OFFSET_S:.1f} s at the start of a median mission, shrinking "
      f"to 0 at the end (mean {MEDIAN_OFFSET_S / 2:.1f} s = {MEDIAN_OFFSET_S / 2 / WINDOW_SEC:.1f} windows)")

# --- 3. exclusion rule: offset / duration >= AUDIO_EXCLUDE_RATIO -> out of CORRECTED_ALIGNMENT (every split) ---
_ratios = {n: p["offset_ratio"] for n, (m, p) in _by_name.items()}
EXCLUDED_MISSIONS = {n: r for n, r in sorted(_ratios.items()) if r >= AUDIO_EXCLUDE_RATIO}
print(f"\n[EXCLUSION] rule: audio_start_offset_s / mission_duration_s >= {AUDIO_EXCLUDE_RATIO} -> excluded from "
      f"CORRECTED_ALIGNMENT only ({len(EXCLUDED_MISSIONS)} missions):")
for _name, _r in EXCLUDED_MISSIONS.items():
    _m, _p = _by_name[_name]
    _s, _lab = mission_home[_m]
    print(f"    EXCLUDED {_name:<30} ratio {_r:.3f} | offset {_p['audio_start_offset_s']:.2f} s of "
          f"{_p['mission_duration_s']:.2f} s | class {_lab} | split {_s}")
print("    nearest kept missions (highest ratios below the threshold): " + ", ".join(
    f"{n} {r:.3f}" for n, r in sorted(((n, r) for n, r in _ratios.items() if r < AUDIO_EXCLUDE_RATIO), key=lambda t: -t[1])[:3]))
EXCLUSION_MATCHES = set(EXCLUDED_MISSIONS) == EXPECTED_EXCLUDED_MISSIONS
if EXCLUSION_MATCHES:
    print(f"[EXCLUSION] ✅ computed exclusion set == the 4 expected missions {sorted(EXPECTED_EXCLUDED_MISSIONS)}")
else:
    print("\n" + "🚨" * 40)
    print(f"🚨 EXCLUSION SET DOES NOT MATCH THE EXPECTED LIST\n🚨   computed but not expected: "
          f"{sorted(set(EXCLUDED_MISSIONS) - EXPECTED_EXCLUDED_MISSIONS)}\n🚨   expected but not computed: "
          f"{sorted(EXPECTED_EXCLUDED_MISSIONS - set(EXCLUDED_MISSIONS))} (ratios: "
          f"{ {n: round(_ratios[n], 3) for n in EXPECTED_EXCLUDED_MISSIONS if n in _ratios} })\n🚨 The computed set is "
          f"what is applied. Check the metadata before trusting this run.")
    print("🚨" * 40 + "\n")
corrected_splits = {s: {label: [m for m in mission_splits[s][label]
                                if corrected_parsed[m] is not None and m.name not in EXCLUDED_MISSIONS]
                        for label in mission_splits[s]} for s in mission_splits}
CORRECTED_N_MISSIONS = sum(len(v) for s in corrected_splits for v in corrected_splits[s].values())
print(f"[CORRECTED MISSIONS] {CORRECTED_N_MISSIONS} missions (expected {EXPECTED_N_MISSIONS['CORRECTED_ALIGNMENT']}) -> "
      + ("✅" if CORRECTED_N_MISSIONS == EXPECTED_N_MISSIONS["CORRECTED_ALIGNMENT"] else "🚨 UNEXPECTED COUNT"))

# --- 4. strongest-64 bins: from CORRECTED's TRAINING missions only (or OLD's frozen bins) ---
_corr_train = [m for label in sorted(corrected_splits["train"]) for m in corrected_splits["train"][label]]
assert set(_corr_train) <= set(split_dirs["train"])
if CORRECTED_BIN_SOURCE == "own_train":
    CORRECTED_BINS = select_strongest_bins([corrected_parsed[m] for m in _corr_train], N_AUDIO_BINS)
else:
    CORRECTED_BINS = OLD_ALIGNMENT["bins"].copy()
_overlap = len(set(CORRECTED_BINS.tolist()) & set(OLD_ALIGNMENT["bins"].tolist()))
CORRECTED_BINS_SAME = CORRECTED_BINS.tolist() == OLD_ALIGNMENT["bins"].tolist()
print(f"[BINS] CORRECTED strongest-{N_AUDIO_BINS} ({CORRECTED_BIN_SOURCE}: {len(_corr_train)} TRAINING missions, "
      f"0 val/test) -> {_overlap}/{N_AUDIO_BINS} shared with OLD -> "
      + ("identical to OLD's bins ✅" if CORRECTED_BINS_SAME else
         f"DIFFERENT: only in CORRECTED {sorted(set(CORRECTED_BINS.tolist()) - set(OLD_ALIGNMENT['bins'].tolist()))}, "
         f"only in OLD {sorted(set(OLD_ALIGNMENT['bins'].tolist()) - set(CORRECTED_BINS.tolist()))}"))

# --- 5. windows ---
corr_windows, corr_window_missions = {}, {}
for split in ("train", "val", "test"):
    corr_windows[split], corr_window_missions[split] = [], []
    for label in sorted(corrected_splits[split]):
        for m in corrected_splits[split][label]:
            _w = slice_windows_CORRECTED_ALIGNMENT(corrected_parsed[m], label, CORRECTED_BINS)
            corr_windows[split].extend(_w)
            corr_window_missions[split].extend([m.name] * len(_w))
    print(f"{split:<5}: {len(corr_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
CORRECTED_FINGERPRINT = {s: split_fingerprint(corr_windows[s]) for s in corr_windows}

# --- 6. datasets: sensor mean/std from CORRECTED's TRAINING split only ---
corr_train_ds = UAVWindowDataset(corr_windows["train"])
corr_val_ds = UAVWindowDataset(corr_windows["val"], corr_train_ds.sensor_mean, corr_train_ds.sensor_std)
corr_test_ds = UAVWindowDataset(corr_windows["test"], corr_train_ds.sensor_mean, corr_train_ds.sensor_std)
assert corr_val_ds.sensor_mean is corr_train_ds.sensor_mean and corr_test_ds.sensor_std is corr_train_ds.sensor_std
print(f"[LEAKAGE CHECK] CORRECTED sensor standardization = CORRECTED training-split stats | mean "
      f"{np.round(corr_train_ds.sensor_mean.ravel(), 3).tolist()}")

# --- 7. audio coverage: how many windows carry all-zero audio (no chunk inside them) ---
def _zero_audio(windows):
    rows = np.stack([np.abs(w["x_a"]).sum(axis=1) == 0 for w in windows])   # (n windows, TA_AUDIO)
    return int(rows.all(axis=1).sum()), int(rows.any(axis=1).sum())
ZERO_AUDIO = {}
for _tag, _wins in (("OLD_ALIGNMENT", OLD_ALIGNMENT["split_windows"]), ("CORRECTED_ALIGNMENT", corr_windows)):
    _all = [w for s in ("train", "val", "test") for w in _wins[s]]
    ZERO_AUDIO[_tag] = _zero_audio(_all)
    print(f"[AUDIO COVERAGE] {_tag:<20}: {ZERO_AUDIO[_tag][0]} of {len(_all)} windows have ALL-zero audio, "
          f"{ZERO_AUDIO[_tag][1]} have some all-zero audio rows")

CORRECTED_ALIGNMENT = {"name": "CORRECTED_ALIGNMENT", "n_missions": CORRECTED_N_MISSIONS, "bins": CORRECTED_BINS,
                       "split_windows": corr_windows, "window_missions": corr_window_missions,
                       "fingerprint": CORRECTED_FINGERPRINT, "train_ds": corr_train_ds, "val_ds": corr_val_ds,
                       "test_ds": corr_test_ds,
                       "split_missions": {s: sorted({n for n in corr_window_missions[s]}) for s in corr_windows},
                       "excluded": EXCLUDED_MISSIONS,
                       "offsets_s": {n: p["audio_start_offset_s"] for n, (m, p) in _by_name.items()}}
del corrected_parsed, _ok, _by_name
gc.collect()

# --- 8. both datasets, side by side ---
DATASETS = {"OLD_ALIGNMENT": OLD_ALIGNMENT, "CORRECTED_ALIGNMENT": CORRECTED_ALIGNMENT}
print("\n" + "=" * 100 + "\nDATASET SIZES — both built, both kept\n" + "=" * 100)
print(f"    {'':<22}" + "".join(f"{n:>24}" for n in DATASETS))
print(f"    {'missions (total)':<22}" + "".join(f"{d['n_missions']:>24}" for d in DATASETS.values()))
for _s in ("train", "val", "test"):
    print(f"    {'missions ' + _s:<22}" + "".join(f"{len(d['split_missions'][_s]):>24}" for d in DATASETS.values()))
for _s in ("train", "val", "test"):
    print(f"    {'windows ' + _s:<22}" + "".join(f"{len(d['split_windows'][_s]):>24}" for d in DATASETS.values()))
print(f"    {'windows (total)':<22}" + "".join(f"{sum(len(v) for v in d['split_windows'].values()):>24}" for d in DATASETS.values()))
for _s in ("train", "test"):
    print(f"    {'class counts ' + _s:<22}" + "".join(f"{str(d['fingerprint'][_s][3]):>24}" for d in DATASETS.values()))
for _name, _r in EXCLUDED_MISSIONS.items():
    print(f"    excluded from CORRECTED: {_name} (split {mission_home[[m for m in mission_home if m.name == _name][0]][0]}, "
          f"ratio {_r:.3f})")


parse train (corrected):   0%|          | 0/69 [00:00<?, ?it/s]

parse val (corrected):   0%|          | 0/14 [00:00<?, ?it/s]

parse test (corrected):   0%|          | 0/16 [00:00<?, ?it/s]

[RAW DATA CHECK] 99 missions parsed by the CORRECTED parser; IMU samples, audio chunks and duration equal OLD's for every mission ✅
[METADATA] recording_started_time is constant within every mission ✅

[OFFSETS] first 8 missions (by name): audio_start_offset_s = (recording_started_time - sensor_t0) / 1e6
    mission                        class split     sensor_t0 (us)   rec_started (us)  offset s    dur s  ratio chunk ms OLD chunk ms NEW
    rosbag2_2021_12_22-09_14_31        0  test          947975062          972352920    24.378    99.30  0.245       18.810       14.192
    rosbag2_2021_12_22-09_17_33        0 train         1130535831         1154353459    23.818    88.12  0.270       18.907       13.797
    rosbag2_2021_12_22-09_21_12        1 train         1349504925         1369333931    19.829   104.98  0.189       17.597       14.273
    rosbag2_2021_12_22-09_23_49        1 train         1514263031         1532148108    17.885   108.04  0.166       17.309       14.444
    rosba

## 3c. v23 reuse checks and the class-level training-data diagnostic
- **CORRECTED is v23's.** The two CORRECTED functions are hashed and compared with v23's source. Mission, window and
  class counts are compared with v23's run. OLD was already checked against v18 in 3a.
- **Training windows per class, OLD → CORRECTED, with % retained.** This tests the alternative explanation for a
  Class 4 recall drop: that CORRECTED simply lost proportionally more Class 4 training data. Training missions and
  val/test windows per class are shown for context.

In [5]:
# ==============================================================================
# v23 REUSE CHECKS (CORRECTED code + data == v23) and the CLASS-LEVEL TRAINING-DATA DIAGNOSTIC
# ==============================================================================
CORRECTED_SOURCE_CHECK = {}
for _name, _obj in (("parse_mission_CORRECTED_ALIGNMENT", parse_mission_CORRECTED_ALIGNMENT),
                    ("slice_windows_CORRECTED_ALIGNMENT", slice_windows_CORRECTED_ALIGNMENT)):
    try:
        _h = _source_sha1(_obj)
        CORRECTED_SOURCE_CHECK[_name] = _h == V23_CORRECTED_SOURCE_SHA1[_name]
        print(f"[CORRECTED == v23 CODE] {_name:<36} sha1 {_h} vs v23 {V23_CORRECTED_SOURCE_SHA1[_name]} -> "
              + ("identical ✅" if CORRECTED_SOURCE_CHECK[_name] else "🚨 DIFFERENT"))
    except (OSError, TypeError) as _e:
        CORRECTED_SOURCE_CHECK[_name] = None
        print(f"[CORRECTED == v23 CODE] {_name:<36} source not readable here ({_e}); the counts below still apply")
CORRECTED_DATA_CHECK = {
    "missions": CORRECTED_ALIGNMENT["n_missions"] == V23_CORRECTED_DATA["n_missions"],
    **{f"{s} missions": len(CORRECTED_ALIGNMENT["split_missions"][s]) == V23_CORRECTED_DATA["missions"][s] for s in ("train", "val", "test")},
    **{f"{s} windows": len(CORRECTED_ALIGNMENT["split_windows"][s]) == V23_CORRECTED_DATA["windows"][s] for s in ("train", "val", "test")},
    **{f"{s} class counts": tuple(CORRECTED_ALIGNMENT["fingerprint"][s][3]) == V23_CORRECTED_DATA["class_counts"][s] for s in ("train", "test")},
    "exclusion set == the 4 expected": EXCLUSION_MATCHES,
    "bins identical to OLD (as in v23)": CORRECTED_BINS_SAME,
}
for _name, _ok in CORRECTED_DATA_CHECK.items():
    print(f"[CORRECTED == v23 DATA] {_name:<36}: {'match ✅' if _ok else '🚨 MISMATCH'}")
DATA_REUSE_OK = (all(OLD_DATA_CHECK.values()) and False not in OLD_SOURCE_CHECK.values()
                 and all(CORRECTED_DATA_CHECK.values()) and False not in CORRECTED_SOURCE_CHECK.values())
print(("✅ Both preprocessing paths are reused exactly as built in v23 (OLD == v18, CORRECTED == v23): same code, same "
       "missions, same windows.") if DATA_REUSE_OK else
      "🚨 A preprocessing path differs from v23 — seed 42 will NOT be reused, and results are not comparable to v23.")


def _class_counts(windows):
    return np.bincount([w["label"] for w in windows], minlength=N_CLASSES)


def _class_missions(data, split):
    return np.bincount([mission_home[m][1] for m in mission_home if m.name in set(data["split_missions"][split])],
                       minlength=N_CLASSES)


TRAIN_COUNTS = {a: _class_counts(DATASETS[a]["split_windows"]["train"]) for a in ALIGNMENTS}
TRAIN_RETAINED_PCT = 100.0 * TRAIN_COUNTS["CORRECTED_ALIGNMENT"] / TRAIN_COUNTS["OLD_ALIGNMENT"]
_others = [c for c in range(N_CLASSES) if c != 4]
C4_RETENTION_GAP_PP = float(TRAIN_RETAINED_PCT[4] - TRAIN_RETAINED_PCT[_others].mean())
C4_DISPROPORTIONATE = C4_RETENTION_GAP_PP <= -DISPROPORTION_PP
_tm = {a: _class_missions(DATASETS[a], "train") for a in ALIGNMENTS}

print("\n" + "█" * 100)
print("█  CLASS-LEVEL TRAINING-DATA DIAGNOSTIC — training windows per class, OLD_ALIGNMENT -> CORRECTED_ALIGNMENT")
print("█" * 100)
print(f"█    {'class':<8}{'OLD windows':>13}{'CORRECTED':>12}{'retained':>11}{'   train missions OLD -> CORRECTED':>36}")
for c in range(N_CLASSES):
    print(f"█    Class {c:<2}{TRAIN_COUNTS['OLD_ALIGNMENT'][c]:>13}{TRAIN_COUNTS['CORRECTED_ALIGNMENT'][c]:>12}"
          f"{TRAIN_RETAINED_PCT[c]:>10.1f}%{_tm['OLD_ALIGNMENT'][c]:>24} -> {_tm['CORRECTED_ALIGNMENT'][c]}")
print(f"█    {'total':<8}{TRAIN_COUNTS['OLD_ALIGNMENT'].sum():>13}{TRAIN_COUNTS['CORRECTED_ALIGNMENT'].sum():>12}"
      f"{100.0 * TRAIN_COUNTS['CORRECTED_ALIGNMENT'].sum() / TRAIN_COUNTS['OLD_ALIGNMENT'].sum():>10.1f}%")
print("█")
print(f"█  >>> CLASS 4 TRAINING WINDOWS: {TRAIN_COUNTS['OLD_ALIGNMENT'][4]} -> {TRAIN_COUNTS['CORRECTED_ALIGNMENT'][4]} = "
      f"{TRAIN_RETAINED_PCT[4]:.1f}% RETAINED  vs  classes 0-3: "
      + ", ".join(f"C{c} {TRAIN_RETAINED_PCT[c]:.1f}%" for c in _others) + f" (mean {TRAIN_RETAINED_PCT[_others].mean():.1f}%) <<<")
print(f"█  >>> Class 4 retention minus the other classes' mean: {C4_RETENTION_GAP_PP:+.1f} percentage points -> "
      + (f"DISPROPORTIONATE Class-4 loss (<= -{DISPROPORTION_PP:.0f} pp): a data-quantity confound is plausible <<<"
         if C4_DISPROPORTIONATE else f"NOT disproportionate (threshold -{DISPROPORTION_PP:.0f} pp): Class 4 lost no more "
         f"training data than the other classes <<<"))
print(f"█  Class 4 rank among the 5 retention rates: {int((TRAIN_RETAINED_PCT < TRAIN_RETAINED_PCT[4]).sum()) + 1} of 5 "
      f"(1 = lowest). Note: the loss is class-weighted by each dataset's OWN training counts, which partly offsets any "
      f"count imbalance.")
print("█" * 100)
for _s in ("val", "test"):
    _cc = {a: _class_counts(DATASETS[a]["split_windows"][_s]) for a in ALIGNMENTS}
    _cm = {a: _class_missions(DATASETS[a], _s) for a in ALIGNMENTS}
    print(f"  (context) {_s:<4} windows per class OLD -> CORRECTED: " + ", ".join(
        f"C{c} {_cc['OLD_ALIGNMENT'][c]}->{_cc['CORRECTED_ALIGNMENT'][c]} ({100.0 * _cc['CORRECTED_ALIGNMENT'][c] / max(_cc['OLD_ALIGNMENT'][c], 1):.0f}%)"
        for c in range(N_CLASSES)) + " | missions per class: " + ", ".join(
        f"C{c} {_cm['OLD_ALIGNMENT'][c]}->{_cm['CORRECTED_ALIGNMENT'][c]}" for c in range(N_CLASSES)))

# --- what the two test-set controls can and cannot change ---
DROPPED_TEST_MISSIONS = sorted(set(OLD_ALIGNMENT["split_missions"]["test"]) - set(CORRECTED_ALIGNMENT["split_missions"]["test"]))
DROPPED_TEST_CLASSES = sorted({mission_home[m][1] for m in mission_home if m.name in DROPPED_TEST_MISSIONS})
MISSION_CONTROL_TOUCHES_C34 = bool({3, 4} & set(DROPPED_TEST_CLASSES))
print(f"\n[CONTROLS] test missions in OLD but not in CORRECTED: {DROPPED_TEST_MISSIONS} (classes {DROPPED_TEST_CLASSES})")
if not MISSION_CONTROL_TOUCHES_C34:
    print("[CONTROLS] ⚠️ none of them is Class 3 or 4 -> the MISSION control (OLD scored on CORRECTED's test missions) keeps "
          "every Class 3/4 test window, so its Class 3/4 recall EQUALS OLD's own BY CONSTRUCTION at every seed. It cannot "
          "detect a Class 3/4 effect of mission composition, because there is none to detect. What differs for "
          "Classes 3/4 is WHICH WINDOWS of the same missions are scored (CORRECTED starts each flight at its audio onset), "
          "which the TIME-MATCHED control isolates.")
_otm = np.array(OLD_ALIGNMENT["window_missions"]["test"])
_ots = OLD_ALIGNMENT["window_starts"]["test"]
_off = CORRECTED_ALIGNMENT["offsets_s"]
TIME_MATCHED_MASK = np.array([(m in set(CORRECTED_ALIGNMENT["split_missions"]["test"])) and (t >= _off[m])
                              for m, t in zip(_otm, _ots)])
MISSION_MASK = np.isin(_otm, CORRECTED_ALIGNMENT["split_missions"]["test"])
_tl = np.array([w["label"] for w in OLD_ALIGNMENT["split_windows"]["test"]])
print(f"[CONTROLS] OLD test windows scored: all {len(_otm)} | mission control {int(MISSION_MASK.sum())} | time-matched "
      f"control {int(TIME_MATCHED_MASK.sum())} (CORRECTED's own test set: {len(CORRECTED_ALIGNMENT['split_windows']['test'])})")
print("[CONTROLS] time-matched windows per class: " + ", ".join(
    f"C{c} {int((TIME_MATCHED_MASK & (_tl == c)).sum())} (CORRECTED {int(CORRECTED_ALIGNMENT['fingerprint']['test'][3][c])})"
    for c in range(N_CLASSES)))


[CORRECTED == v23 CODE] parse_mission_CORRECTED_ALIGNMENT    sha1 49d4b2f5a1359025 vs v23 49d4b2f5a1359025 -> identical ✅
[CORRECTED == v23 CODE] slice_windows_CORRECTED_ALIGNMENT    sha1 8cf47da958eda742 vs v23 8cf47da958eda742 -> identical ✅
[CORRECTED == v23 DATA] missions                            : match ✅
[CORRECTED == v23 DATA] train missions                      : match ✅
[CORRECTED == v23 DATA] val missions                        : match ✅
[CORRECTED == v23 DATA] test missions                       : match ✅
[CORRECTED == v23 DATA] train windows                       : match ✅
[CORRECTED == v23 DATA] val windows                         : match ✅
[CORRECTED == v23 DATA] test windows                        : match ✅
[CORRECTED == v23 DATA] train class counts                  : match ✅
[CORRECTED == v23 DATA] test class counts                   : match ✅
[CORRECTED == v23 DATA] exclusion set == the 4 expected     : match ✅
[CORRECTED == v23 DATA] bins identical to OLD (as in v23

## 4. Model definition — v22 Condition D
This is the v22 model cell, unchanged apart from the probe at the end. `AeroSyncMamba(htt_time=False, laca_lag=False)`
is Condition D:
- HTT returns `norm(z + e_m)`.
- LACA adds `0.0 · LN_grid(g(Δτ))`, which is exactly 0.

Both temporal modules are still built. That keeps the initial weights and the RNG stream identical to v18/v22 seed 42
(`5850841e699f` / `5641b401720d`). Both runs, OLD and CORRECTED, start from these same weights.

In [6]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention; lag bias beta * LN_grid(g(Δτ)), beta = 1.0 when
#    LACA_LAG_BIAS_ENABLED else 0.0; 4 heads). v22: HTT's time embedding is switchable too.
# v23: only Condition D is built (HTT-time OFF, LACA-lag OFF) — the classes below are v22's, unchanged.
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED)
print(f"Model (Condition {CONDITION}): {sum(p.numel() for p in _probe.parameters()):,} params | fusion blocks: "
      f"{type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | "
      f"HTT time embedding {'ON' if _probe.htt.time_enabled else 'OFF'} | LACA beta {_probe.laca.laca_as.beta} / "
      f"{_probe.laca.laca_sa.beta} | backbone: {FUSION_BACKBONE}")
assert not _probe.htt.time_enabled and _probe.laca.laca_as.beta == 0.0 and _probe.laca.laca_sa.beta == 0.0
del _probe


Model (Condition D): 1,688,788 params | fusion blocks: MambaBlock x 2 (874,496 params) | HTT time embedding OFF | LACA beta 0.0 / 0.0 | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 5. Loss, metrics and the training function
The loss and optimizer are v22's, unchanged: class-weighted CE + 0.1 InfoNCE + 0.1 KL, AdamW 1e-4 with cosine decay,
20 epochs, and the best checkpoint by validation macro-F1.

The only change is that `train_one_run(seed, alignment)` takes its data from one of the two dataset **bundles**. A
bundle holds that dataset's loaders, training-set class weights and probe batches. For OLD_ALIGNMENT these objects are
built exactly as v22 built its globals, so the RNG stream is v22's.

Every run checks:
- initial weights and RNG, against that seed's reference (`SEED_INIT_REFERENCE`, shared by both alignments);
- both temporal switches, exactly 0, before training, at every epoch and at the best checkpoint;
- after training, that predictions do not change when every τ is shifted by +0.3. Condition D must ignore timestamps.

In [7]:
# ==============================================================================
# v24: unchanged from v23 except the pairing reference, which is now per seed (SEED_INIT_REFERENCE).
# TRAINING — loss (unchanged), metrics (4 standard + confusion matrix + Class 3/4 recall), switch checks and
# train_one_run(seed, alignment). v22's functions, with the dataset passed in as a bundle instead of globals.
# ==============================================================================
from sklearn.metrics import confusion_matrix


class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def evaluate_probs(y_true, y_prob):
    """4 standard metrics + full 5x5 confusion matrix + per-class recall + Class 3/4 recall + 3->4 / 4->3 counts."""
    cm = confusion_matrix(y_true, y_prob.argmax(axis=1), labels=list(range(N_CLASSES)))
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**compute_metrics(y_true, y_prob), "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3])}


def print_confusion(ev, tag):
    cm = ev["confusion_matrix"]
    print(f"[CONFUSION {tag}] rows = TRUE severity, cols = PREDICTED severity (test set, {int(cm.sum())} windows)")
    print("            " + "".join(f"pred {j:>2} " for j in range(N_CLASSES)) + "|  recall")
    for i in range(N_CLASSES):
        print(f"   true {i:>2} " + "".join(f"{int(cm[i, j]):>8}" for j in range(N_CLASSES)) + f"  |  {ev[f'recall_c{i}']:.4f}")
    print(f"[CLASS 3/4 {tag}] Class 3 recall {ev['class3_recall']:.4f} | Class 4 recall {ev['class4_recall']:.4f} | "
          f"3->4 (true 3, pred 4) {ev['err_3to4']} | 4->3 (true 4, pred 3) {ev['err_4to3']}")


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader, tau_shift=0.0):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order.
    tau_shift != 0 adds the shift to BOTH tau_a and tau_s (used only for the Condition D invariance check)."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta + tau_shift, ts + tau_shift) if tau_shift else model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]


def make_bundle(data):
    """Loaders, class weights and probe batches for one dataset — built exactly as v22 built its globals."""
    tr = data["train_ds"]
    return {**data,
            "train_loader": DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS),
            "val_loader": DataLoader(data["val_ds"], batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "test_loader": DataLoader(data["test_ds"], batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "class_weights": train_class_weights(data["split_windows"]["train"]).to(device),
            "test_labels": np.array([w["label"] for w in data["split_windows"]["test"]], dtype=np.int64),
            "test_missions": np.array(data["window_missions"]["test"]),
            # fixed probe batches, read by index (no DataLoader, no RNG)
            "lag_probe_taus": tuple(torch.stack([tr[i][k] for i in range(min(BATCH_SIZE, len(tr)))]) for k in ("tau_a", "tau_s")),
            "switch_probe_idx": np.linspace(0, len(tr) - 1, BATCH_SIZE).astype(int)}


BUNDLES = {name: make_bundle(DATASETS[name]) for name in ALIGNMENTS}
for name, b in BUNDLES.items():
    assert len(b["test_labels"]) == len(b["test_missions"]) == len(b["test_ds"])
    print(f"[BUNDLE] {name:<20}: train {len(b['train_ds'])} / val {len(b['val_ds'])} / test {len(b['test_ds'])} windows | "
          f"class weights {np.round(b['class_weights'].cpu().numpy(), 4).tolist()}")


def lag_bias_contribution(model, bundle):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores."""
    ta, ts = (t.to(device) for t in bundle["lag_probe_taus"])
    out = {}
    with torch.no_grad():
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            contrib = laca.beta * laca.lag_bias_term(tau_m, tau_n)
            out[name] = {"beta": laca.beta, "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all())}
    return out


def htt_time_contribution(model, bundle):
    """What HTT's time embedding contributes on a fixed real batch (eval mode, RNG untouched):
    term = max|HTT(z, tau) - norm(z + e_m)| and tau_sens = max|HTT(z, tau) - HTT(z, tau + 0.3)|, per modality."""
    items = [bundle["train_ds"][int(i)] for i in bundle["switch_probe_idx"]]
    xa, xs, ta, ts = (torch.stack([it[k] for it in items]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    was_training = model.training
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    out = {}
    with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
        model.eval()
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        for name, z, tau, idx in (("audio", za, ta, 0), ("sensor", zs, ts, 1)):
            e_m = model.htt.modality_embeddings(torch.tensor(idx, device=device)).view(1, 1, model.d_model)
            actual = model.htt(z, tau, modality_idx=idx)
            out[name] = {"term": float((actual - model.htt.norm(z + e_m)).abs().max()),
                         "tau_sens": float((actual - model.htt(z, tau + 0.3, modality_idx=idx)).abs().max())}
    model.train(was_training)
    return out


def check_switches(model, bundle, tag, when):
    """Condition D: both temporal contributions must be EXACTLY 0."""
    h = htt_time_contribution(model, bundle)
    lag = lag_bias_contribution(model, bundle)
    h_max = max(max(v["term"], v["tau_sens"]) for v in h.values())
    lag_max = max(r["contrib_absmax"] for r in lag.values())
    assert h_max == 0.0, f"HTT time embedding is OFF but contributes {h_max!r}"
    assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"LACA lag OFF but {lag}"
    print(f"{tag} [SWITCH CHECK {when}] ✅ HTT time term {h_max!r}, LACA lag term {lag_max!r} — both EXACTLY 0")


def train_one_run(seed: int, alignment: str):
    """Train Condition D with `seed` on dataset `alignment`, evaluate on that dataset's test set.
    Returns (trained model, result dict incl. test probabilities and confusion matrix)."""
    t0 = time.time()
    bundle = BUNDLES[alignment]
    tag = f"[{alignment} | {CONDITION} seed={seed}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model)
    ref_init, ref_rng = SEED_INIT_REFERENCE[seed]
    paired_ok = init_sha1[:12] == ref_init and rng_sha1[:12] == ref_rng
    print(f"{tag} HTT_TIME_EMBED_ENABLED = {model.htt.time_enabled} | LACA_LAG_BIAS_ENABLED = {model.laca_lag_enabled} "
          f"(beta {model.laca.laca_as.beta}) | fusion {type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params) | "
          f"train {len(bundle['train_ds'])} / val {len(bundle['val_ds'])} / test {len(bundle['test_ds'])} windows")
    print(f"{tag} [PAIRING] initial weights {init_sha1[:12]} vs reference {ref_init} | RNG {rng_sha1[:12]} vs reference "
          f"{ref_rng} -> " + ("✅ identical start (same for OLD and CORRECTED of this seed)" if paired_ok else
                              "🚨 MISMATCH — this seed's two alignments are not cleanly paired"))
    check_switches(model, bundle, tag, "before training")

    criterion = AeroSyncLoss(bundle["class_weights"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for batch in tqdm(bundle["train_loader"], desc=f"{alignment} seed {seed} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        lag = lag_bias_contribution(model, bundle)
        assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"lag leak at epoch {epoch}"

        val_probs, val_labels = predict_probs(model, bundle["val_loader"])
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(bundle["train_loader"])
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | lag term 0 ✅ | train loss {running['loss'] / n:.4f} (cls "
              f"{running['loss_cls'] / n:.4f}, align {running['loss_align'] / n:.4f}, cons {running['loss_cons'] / n:.4f}) "
              f"| val macro-F1 {val_f1:.4f}{'  * best' if improved else ''}")

    model.load_state_dict(best_state)
    check_switches(model, bundle, tag, f"best checkpoint, epoch {best_epoch}")
    test_probs, test_labels = predict_probs(model, bundle["test_loader"])
    assert np.array_equal(test_labels, bundle["test_labels"])
    ev = evaluate_probs(test_labels, test_probs)
    shifted_probs, _ = predict_probs(model, bundle["test_loader"], tau_shift=0.3)
    tau_invariant = bool(np.array_equal(shifted_probs, test_probs))
    print(f"{tag} [TAU INVARIANCE] test predictions with every tau_a / tau_s shifted by +0.3: "
          + ("✅ bit-identical — Condition D uses no timestamps; only the audio CONTENT of each window differs "
             "between OLD and CORRECTED" if tau_invariant else "🚨 CHANGED — a timestamp path is active in Condition D"))
    run_name = f"{alignment}_{CONDITION}_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "alignment": alignment, "condition": CONDITION, "seed": seed,
        **ev,
        "n_missions": bundle["n_missions"],
        "n_train_windows": len(bundle["train_ds"]),
        "n_test_windows": len(test_labels),
        "n_test_missions": len(bundle["split_missions"]["test"]),
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_init": paired_ok,
        "tau_invariant": tau_invariant,
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model, result


[BUNDLE] OLD_ALIGNMENT       : train 4934 / val 954 / test 1130 windows | class weights [1.0892000198364258, 0.9674999713897705, 0.9829000234603882, 0.9789999723434448, 0.9908000230789185]
[BUNDLE] CORRECTED_ALIGNMENT : train 3737 / val 686 / test 815 windows | class weights [1.0662000179290771, 0.9769999980926514, 1.0585999488830566, 0.9388999938964844, 0.9718999862670898]


## 6. Driver — seed-42 reuse decision, then 5 paired seeds (OLD, then CORRECTED)
**Seed 42 is reused only when all of these hold:**
- v23's saved test probabilities and labels are found for both alignments;
- the labels equal this notebook's test windows, window for window;
- the probabilities reproduce v23's logged metrics exactly (|Δ| < 1e-12) and its confusion matrices;
- both datasets pass the v23 checks;
- a freshly built seed-42 model has v23's initial weights and RNG hash.

Otherwise seed 42 is retrained and compared with v23 (±0.0005).

**After every OLD model**, both controls are scored: the mission control and the time-matched control. Each run gets
one CSV row per evaluation set, the CSV is written after every run, and GPU memory is freed. All confusion matrices are
printed.

In [ ]:
# ==============================================================================
# DRIVER — seed-42 reuse decision; then per seed OLD -> CORRECTED; controls on every OLD model; CSV after every run.
# ==============================================================================
import glob

if RESULTS_CSV.exists():
    RESULTS_CSV.rename(RESULTS_CSV.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))
RESULTS_CSV_FIELDS = (["slot", "alignment", "condition", "seed", "eval_set", "source"] + METRICS
                      + ["class3_recall", "class4_recall", "err_3to4", "err_4to3"] + [f"recall_c{c}" for c in range(N_CLASSES)]
                      + ["confusion_matrix", "n_missions", "n_train_windows", "n_test_windows", "n_test_missions",
                         "best_val_macro_f1", "best_epoch", "init_sha1", "rng_sha1", "paired_init", "tau_invariant",
                         "train_seconds", "finished_at"])


def log_row(slot, result, eval_set, source):
    write_header = not RESULTS_CSV.exists()
    row = {k: result[k] for k in RESULTS_CSV_FIELDS if k in result}
    row.update(slot=slot, eval_set=eval_set, source=source,
               confusion_matrix=json.dumps(np.asarray(result["confusion_matrix"]).tolist()),
               finished_at=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def control_eval(result, mask, name):
    """The same OLD model's test probabilities, scored on a subset of OLD's test windows."""
    return {**{k: v for k, v in result.items() if k not in ("test_probs", "test_labels")},
            **evaluate_probs(result["test_labels"][mask], result["test_probs"][mask]),
            "n_test_windows": int(mask.sum()), "n_test_missions": len(set(BUNDLES["OLD_ALIGNMENT"]["test_missions"][mask])),
            "eval_set": name}


def _find_v23(name):
    for pattern in V23_PROBS_GLOBS:
        hits = sorted(glob.glob(pattern.format(name=name), recursive=True))
        if hits:
            return hits[0]
    return None


# --- seed-42 reuse decision ---
REUSE_42, REUSE_42_REASON, REUSED = False, [], {}
if REUSE_SEED42 != "auto":
    REUSE_42_REASON.append(f"REUSE_SEED42 = {REUSE_SEED42!r}")
elif not DATA_REUSE_OK:
    REUSE_42_REASON.append("a preprocessing path differs from v23")
else:
    set_all_seeds(42)
    _m = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED)
    _init = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes() for v in _m.state_dict().values())).hexdigest()[:12]
    _rng = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()[:12]
    del _m
    if (_init, _rng) != SEED_INIT_REFERENCE[42]:
        REUSE_42_REASON.append(f"seed-42 initial weights / RNG {(_init, _rng)} != v23 {SEED_INIT_REFERENCE[42]}")
    for a in ALIGNMENTS:
        p_path, l_path = _find_v23(f"{a}_{CONDITION}_seed42_test_probs.npy"), _find_v23(f"{a}_{CONDITION}_seed42_test_labels.npy")
        if p_path is None or l_path is None:
            REUSE_42_REASON.append(f"v23 test probabilities for {a} not found (attach v23's output: v23_test_probs/)")
            continue
        probs, labels = np.load(p_path), np.load(l_path)
        if not np.array_equal(labels, BUNDLES[a]["test_labels"]):
            REUSE_42_REASON.append(f"{a}: v23 test labels differ from this notebook's test windows")
            continue
        ev = evaluate_probs(labels, probs)
        ref = V23_SEED42[a]
        worst = max(abs(ev[k] - ref[k]) for k in METRICS + ["class3_recall", "class4_recall"])
        cm_ok = np.array_equal(ev["confusion_matrix"], np.array(ref["confusion_matrix"]))
        print(f"[REUSE CHECK] {a}: {p_path} -> max |metric - v23| {worst:.1e}, confusion matrix "
              f"{'identical' if cm_ok else 'DIFFERENT'}")
        if worst >= 1e-12 or not cm_ok:
            REUSE_42_REASON.append(f"{a}: v23 probabilities do not reproduce v23's logged metrics")
            continue
        REUSED[a] = {"alignment": a, "condition": CONDITION, "seed": 42, **ev, "test_probs": probs, "test_labels": labels,
                     "n_missions": BUNDLES[a]["n_missions"], "n_train_windows": len(BUNDLES[a]["train_ds"]),
                     "n_test_windows": len(labels), "n_test_missions": len(BUNDLES[a]["split_missions"]["test"]),
                     "best_val_macro_f1": ref["best_val_macro_f1"], "best_epoch": ref["best_epoch"],
                     "init_sha1": SEED_INIT_REFERENCE[42][0], "rng_sha1": SEED_INIT_REFERENCE[42][1],
                     "paired_init": True, "tau_invariant": True}   # both asserted/checked in v23's run
    REUSE_42 = not REUSE_42_REASON and set(REUSED) == set(ALIGNMENTS)
print("\n" + "#" * 100)
if REUSE_42:
    print("#  SEED 42: REUSED from v23 (NOT retrained) — v23's saved test probabilities reproduce v23's logged metrics and "
          "confusion matrices exactly for both alignments, on identical test windows, with identical data and initial "
          f"weights. Runs to train: {len(RUN_ORDER) - 2}.")
else:
    print("#  SEED 42: RETRAINED — reuse not possible: " + "; ".join(REUSE_42_REASON or ["unknown"]))
    print(f"#  The retrained runs are compared with v23's logged seed-42 results (±{SAME_SEED_TOLERANCE}). Runs to train: "
          f"{len(RUN_ORDER)}.")
print("#" * 100)

run_results, controls, sources = {}, {}, {}
SEED42_REPRO = {}
n_trained, t_driver = 0, time.time()
for slot, (seed, alignment) in enumerate(RUN_ORDER, start=1):
    print("\n" + "=" * 100 + f"\nSLOT {slot}/{len(RUN_ORDER)} — seed {seed} | {alignment} | Condition {CONDITION}\n" + "=" * 100)
    model = None
    if seed == 42 and REUSE_42:
        result, source = REUSED[alignment], "reused v23 (not retrained)"
    else:
        model, result = train_one_run(seed, alignment)
        source = "trained in v24"
        n_trained += 1
    run_results[(seed, alignment)], sources[(seed, alignment)] = result, source
    print(f"[{alignment} seed={seed}] ({source}) TEST: accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | "
          f"balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f} | best epoch "
          f"{result['best_epoch']}")
    print_confusion(result, f"{alignment} seed={seed}")
    log_row(slot, result, "own test set", source)

    if seed == 42 and not REUSE_42:
        ref = V23_SEED42[alignment]
        delta = max(abs(result[k] - ref[k]) for k in METRICS)
        SEED42_REPRO[alignment] = (delta, np.array_equal(result["confusion_matrix"], np.array(ref["confusion_matrix"])))
        print(f"[SEED-42 REPRODUCTION] {alignment} retrained vs v23: max |delta| {delta:.1e}, confusion matrix "
              f"{'identical' if SEED42_REPRO[alignment][1] else 'DIFFERENT'} -> "
              + ("✅ reproduces v23" if delta <= SAME_SEED_TOLERANCE else "🚨 does NOT reproduce v23"))

    if alignment == "OLD_ALIGNMENT":
        controls[(seed, "mission")] = control_eval(result, MISSION_MASK, "OLD model on CORRECTED's test missions")
        controls[(seed, "time")] = control_eval(result, TIME_MATCHED_MASK,
                                                "OLD model on CORRECTED's test missions, windows from the audio onset")
        for kind in ("mission", "time"):
            c = controls[(seed, kind)]
            print(f"[CONTROL {kind.upper():<7} seed={seed}] {c['eval_set']} ({c['n_test_windows']} windows): macro-F1 "
                  f"{c['macro_f1']:.4f} | Class 3 recall {c['class3_recall']:.4f} ({c['class3_recall'] - result['class3_recall']:+.4f} "
                  f"vs OLD) | Class 4 recall {c['class4_recall']:.4f} ({c['class4_recall'] - result['class4_recall']:+.4f} vs OLD)")
            print_confusion(c, f"CONTROL {kind} seed={seed}")
            log_row(slot, c, c["eval_set"], source)
        if seed == 42:
            _m42 = controls[(42, "mission")]
            _ok = all(abs(_m42[k] - V23_SEED42["OLD_ON_CORRECTED_MISSIONS"][k]) <= SAME_SEED_TOLERANCE
                      for k in ("macro_f1", "class3_recall", "class4_recall"))
            print(f"[SEED-42 CONTROL vs v23] mission control macro-F1 {_m42['macro_f1']:.4f} vs v23 "
                  f"{V23_SEED42['OLD_ON_CORRECTED_MISSIONS']['macro_f1']:.4f} -> {'✅ matches' if _ok else '🚨 differs'}")

    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elapsed = time.time() - t_driver
    remaining = sum(1 for s, a in RUN_ORDER[slot:] if not (s == 42 and REUSE_42))
    print(f"\n>>> {slot}/{len(RUN_ORDER)} slots complete | {n_trained} trained | elapsed {elapsed / 60:.1f} min | est. remaining "
          f"{elapsed / max(n_trained, 1) * remaining / 60:.1f} min ({remaining} runs to train)")

print(f"\nAll {len(RUN_ORDER)} slots done: {n_trained} trained, {len(RUN_ORDER) - n_trained} reused, in "
      f"{(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV}")



####################################################################################################
#  SEED 42: RETRAINED — reuse not possible: v23 test probabilities for OLD_ALIGNMENT not found (attach v23's output: v23_test_probs/); v23 test probabilities for CORRECTED_ALIGNMENT not found (attach v23's output: v23_test_probs/)
#  The retrained runs are compared with v23's logged seed-42 results (±0.0005). Runs to train: 10.
####################################################################################################

SLOT 1/10 — seed 42 | OLD_ALIGNMENT | Condition D
[OLD_ALIGNMENT | D seed=42] HTT_TIME_EMBED_ENABLED = False | LACA_LAG_BIAS_ENABLED = False (beta 0.0) | fusion MambaBlock (874,496 params) | train 4934 / val 954 / test 1130 windows
[OLD_ALIGNMENT | D seed=42] [PAIRING] initial weights 5850841e699f vs reference 5850841e699f | RNG 5641b401720d vs reference 5641b401720d -> ✅ identical start (same for OLD and CORRECTED of this seed)
[OLD_ALIGNMENT | D seed=42] [SWIT

OLD_ALIGNMENT seed 42 epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[OLD_ALIGNMENT | D seed=42] epoch 01/20 | lag term 0 ✅ | train loss 1.4149 (cls 1.0677, align 3.1039, cons 0.3679) | val macro-F1 0.8307  * best


OLD_ALIGNMENT seed 42 epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[OLD_ALIGNMENT | D seed=42] epoch 02/20 | lag term 0 ✅ | train loss 0.9230 (cls 0.5942, align 2.6671, cons 0.6210) | val macro-F1 0.6824


...

[CORRECTED_ALIGNMENT | D seed=99] epoch 18/20 | lag term 0 ✅ | train loss 0.4104 (cls 0.1496, align 2.1825, cons 0.4254) | val macro-F1 0.9212


CORRECTED_ALIGNMENT seed 99 epoch 19/20:   0%|          | 0/117 [00:00<?, ?it/s]

[CORRECTED_ALIGNMENT | D seed=99] epoch 19/20 | lag term 0 ✅ | train loss 0.3852 (cls 0.1260, align 2.1643, cons 0.4284) | val macro-F1 0.9130


CORRECTED_ALIGNMENT seed 99 epoch 20/20:   0%|          | 0/117 [00:00<?, ?it/s]

[CORRECTED_ALIGNMENT | D seed=99] epoch 20/20 | lag term 0 ✅ | train loss 0.3864 (cls 0.1271, align 2.1653, cons 0.4275) | val macro-F1 0.9144
[CORRECTED_ALIGNMENT | D seed=99] [SWITCH CHECK best checkpoint, epoch 10] ✅ HTT time term 0.0, LACA lag term 0.0 — both EXACTLY 0
[CORRECTED_ALIGNMENT | D seed=99] [TAU INVARIANCE] test predictions with every tau_a / tau_s shifted by +0.3: ✅ bit-identical — Condition D uses no timestamps; only the audio CONTENT of each window differs between OLD and CORRECTED
[CORRECTED_ALIGNMENT seed=99] (trained in v24) TEST: accuracy 0.8945 | macro-F1 0.8846 | balanced accuracy 0.8758 | macro-AUROC 0.9895 | best epoch 10
[CONFUSION CORRECTED_ALIGNMENT seed=99] rows = TRUE severity, cols = PREDICTED severity (test set, 815 windows)
            pred  0 pred  1 pred  2 pred  3 pred  4 |  recall
   true  0      213       1       0       0       0  |  0.9953
   true  1        1     152       3       0       0  |  0.9744
   true  2        0       2      77      27

## 7. Paired statistics and the verdict
**Per-seed table.** The table covers macro-F1 and Class 3/4 recall for:
- OLD;
- the mission control;
- the time-matched control;
- CORRECTED.

**Paired statistics.** For each of the 3 metrics, the paired difference is CORRECTED − OLD. The table reports the
mean, SD, wins out of 5 and the number of seeds in seed 42's direction. The primary test is a paired t-test. An exact
Wilcoxon signed-rank test is secondary: with n = 5 its minimum two-sided p is 0.0625, so it can never reach 0.05.

**Decomposition.** The time-matched control splits each difference into two paired parts:
- **flight-phase effect**: time-matched OLD − OLD, i.e. the same model scored without the pre-onset windows;
- **remainder**: CORRECTED − time-matched OLD. This remainder mixes correct audio alignment with training on the
  trimmed data.

All 10 confusion matrices are shown side by side.

**Verdict.** Answers to (a)–(d). A metric counts as **replicated** when the paired t-test gives p < 0.05 **and** at
least 4 of 5 seeds move in seed 42's direction.

In [ ]:
# ==============================================================================
# PAIRED ANALYSIS — per-seed table, paired stats (macro-F1, Class 3 recall, Class 4 recall), decomposition, verdict
# ==============================================================================
from scipy import stats

KEYS = [("macro-F1", "macro_f1"), ("Class 3 recall", "class3_recall"), ("Class 4 recall", "class4_recall")]
OLD = {s: run_results[(s, "OLD_ALIGNMENT")] for s in SEEDS}
CORR = {s: run_results[(s, "CORRECTED_ALIGNMENT")] for s in SEEDS}
MIS = {s: controls[(s, "mission")] for s in SEEDS}
TIM = {s: controls[(s, "time")] for s in SEEDS}

print("=" * 124)
print(f"PER-SEED RESULTS — Condition {CONDITION} (source: " + ", ".join(f"{s}: {'reused' if sources[(s, 'OLD_ALIGNMENT')].startswith('reused') else 'trained'}" for s in SEEDS) + ")")
print("=" * 124)
for label, k in KEYS:
    print(f"  {label}")
    print(f"    {'seed':>6}{'OLD':>10}{'mission ctl':>13}{'time-matched':>14}{'CORRECTED':>11}{'CORR-OLD':>11}"
          f"{'ctl-OLD (mission)':>19}{'time-OLD (phase)':>18}{'CORR-time':>11}")
    for s in SEEDS:
        print(f"    {s:>6}{OLD[s][k]:>10.4f}{MIS[s][k]:>13.4f}{TIM[s][k]:>14.4f}{CORR[s][k]:>11.4f}{CORR[s][k] - OLD[s][k]:>+11.4f}"
              f"{MIS[s][k] - OLD[s][k]:>+19.4f}{TIM[s][k] - OLD[s][k]:>+18.4f}{CORR[s][k] - TIM[s][k]:>+11.4f}")
    print(f"    {'mean':>6}{np.mean([OLD[s][k] for s in SEEDS]):>10.4f}{np.mean([MIS[s][k] for s in SEEDS]):>13.4f}"
          f"{np.mean([TIM[s][k] for s in SEEDS]):>14.4f}{np.mean([CORR[s][k] for s in SEEDS]):>11.4f}")
    print(f"    {'SD':>6}{np.std([OLD[s][k] for s in SEEDS], ddof=1):>10.4f}{np.std([MIS[s][k] for s in SEEDS], ddof=1):>13.4f}"
          f"{np.std([TIM[s][k] for s in SEEDS], ddof=1):>14.4f}{np.std([CORR[s][k] for s in SEEDS], ddof=1):>11.4f}")


def paired(a, b):
    """a - b per seed -> mean, SD, wins (a > b), same sign as seed 42, paired t p, exact Wilcoxon p."""
    d = np.array([a[s] - b[s] for s in SEEDS])
    s42 = np.sign(d[SEEDS.index(42)])
    if d.std() == 0:   # all 5 differences identical: the t statistic is undefined
        t_p = 1.0 if d[0] == 0 else 0.0
    else:
        t_p = float(stats.ttest_rel([a[s] for s in SEEDS], [b[s] for s in SEEDS]).pvalue)
    if np.any(d != 0):
        try:
            w_p = float(stats.wilcoxon(d, alternative="two-sided", method="exact").pvalue)
        except TypeError:   # older scipy: the keyword was `mode`
            w_p = float(stats.wilcoxon(d, alternative="two-sided", mode="exact").pvalue)
    else:
        w_p = 1.0
    return {"diffs": d, "mean": float(d.mean()), "sd": float(d.std(ddof=1)), "wins": int((d > 0).sum()),
            "losses": int((d < 0).sum()), "same_as_42": int((np.sign(d) == s42).sum()) if s42 != 0 else 0,
            "t_p": t_p, "w_p": w_p, "sign42": s42}


PAIRED, PHASE, REMAINDER, MISSION_CTL = {}, {}, {}, {}
print("\n" + "=" * 124 + "\nPAIRED STATISTICS over 5 seeds — paired t-test (primary), exact Wilcoxon (secondary; min p = 0.0625 at n = 5)\n" + "=" * 124)
print(f"    {'comparison':<44}{'metric':<16}{'mean':>9}{'SD':>8}{'wins':>6}{'same dir as 42':>16}{'t-test p':>10}{'Wilcoxon p':>12}")
for label, k in KEYS:
    for tag, store, a, b in (("CORRECTED - OLD", PAIRED, CORR, OLD),
                             ("time-matched OLD - OLD (flight phase)", PHASE, TIM, OLD),
                             ("CORRECTED - time-matched OLD (remainder)", REMAINDER, CORR, TIM),
                             ("mission control - OLD", MISSION_CTL, MIS, OLD)):
        r = store[k] = paired({s: a[s][k] for s in SEEDS}, {s: b[s][k] for s in SEEDS})
        print(f"    {tag:<44}{label:<16}{r['mean']:>+9.4f}{r['sd']:>8.4f}{r['wins']:>4}/5{r['same_as_42']:>14}/5"
              f"{r['t_p']:>10.4f}{r['w_p']:>12.4f}")
    print()

print("[CONFUSION MATRICES — all runs] rows = TRUE severity, cols = PREDICTED severity")
for s in SEEDS:
    print(f"  seed {s:<5} {'OLD_ALIGNMENT':<45}{'CORRECTED_ALIGNMENT':<45}")
    for i in range(N_CLASSES):
        print("             " + "     ".join(f"t{i} " + "".join(f"{int(r['confusion_matrix'][i, j]):>7}" for j in range(N_CLASSES))
                                            for r in (OLD[s], CORR[s])))


def replicated(r):
    return r["t_p"] < PAIRED_ALPHA and r["same_as_42"] >= REPLICATION_MIN_SAME_SIGN


C4, C3, F1 = PAIRED["class4_recall"], PAIRED["class3_recall"], PAIRED["macro_f1"]
OLD_C4_SD = float(np.std([OLD[s]["class4_recall"] for s in SEEDS], ddof=1))
OLD_C3_SD = float(np.std([OLD[s]["class3_recall"] for s in SEEDS], ddof=1))
print("\n" + "#" * 124)
print("#  VERDICT")
print("#" * 124)
print(f"#  macro-F1: CORRECTED - OLD mean {F1['mean']:+.4f} (SD {F1['sd']:.4f}, wins {F1['wins']}/5, t p {F1['t_p']:.4f}) | "
      f"|mean| {'>' if abs(F1['mean']) > SEED_TO_SEED_STD_F1 else '<='} the 0.0160 seed-to-seed floor")
for letter, name, r, sd in (("a", "Class 4 recall DROP", C4, OLD_C4_SD), ("b", "Class 3 recall GAIN", C3, OLD_C3_SD)):
    per_seed = ", ".join(f"{s}: {d:+.4f}" for s, d in zip(SEEDS, r["diffs"]))
    print(f"#\n#  ({letter}) {name} — per seed [{per_seed}]")
    print(f"#      mean {r['mean']:+.4f}, SD {r['sd']:.4f}, {r['same_as_42']}/5 seeds in seed 42's direction, paired t p = "
          f"{r['t_p']:.4f} (Wilcoxon p = {r['w_p']:.4f}) | OLD's own seed-to-seed SD of this recall: {sd:.4f}")
    if replicated(r):
        print(f"#      -> REPLICATES across 5 seeds (p < {PAIRED_ALPHA}, {r['same_as_42']}/5 same direction). Not a single-seed artifact.")
    elif r["same_as_42"] >= REPLICATION_MIN_SAME_SIGN:
        print(f"#      -> CONSISTENT IN DIRECTION ({r['same_as_42']}/5) but NOT significant (p = {r['t_p']:.4f}): suggestive, not established.")
    else:
        print(f"#      -> DOES NOT REPLICATE ({r['same_as_42']}/5 in seed 42's direction, p = {r['t_p']:.4f}): the seed-42 swing looks "
              f"like a single-seed artifact.")

print(f"#\n#  (c) Class-4 training-data explanation: Class 4 kept {TRAIN_RETAINED_PCT[4]:.1f}% of its OLD training windows vs "
      f"{TRAIN_RETAINED_PCT[_others].mean():.1f}% for classes 0-3 ({C4_RETENTION_GAP_PP:+.1f} pp; "
      + ", ".join(f"C{c} {TRAIN_RETAINED_PCT[c]:.1f}%" for c in range(N_CLASSES)) + ").")
if C4_DISPROPORTIONATE:
    print("#      -> Class 4 lost DISPROPORTIONATELY more training data: a data-quantity confound CAN explain a Class 4 drop. "
          "The alignment effect cannot be separated from it here.")
else:
    print("#      -> Class 4 did NOT lose disproportionately more training data, and the loss is class-weighted by each "
          "dataset's own counts. So proportionally less Class 4 training data does NOT explain "
          + ("the replicated Class 4 drop: the effect persists after accounting for it." if replicated(C4) else
             "the seed-42 drop (which in any case did not replicate)."))

print(f"#\n#  (d) Mission control (OLD model on CORRECTED's test missions), Class 3/4 recall minus OLD per seed: "
      f"C3 [{', '.join(f'{d:+.4f}' for d in MISSION_CTL['class3_recall']['diffs'])}] | "
      f"C4 [{', '.join(f'{d:+.4f}' for d in MISSION_CTL['class4_recall']['diffs'])}]")
_mc_zero = all(np.all(MISSION_CTL[k]["diffs"] == 0) for k in ("class3_recall", "class4_recall"))
if _mc_zero and not MISSION_CONTROL_TOUCHES_C34:
    print(f"#      -> NO swing at any of the 5 seeds, with no seed-to-seed variability. This is BY CONSTRUCTION: the only dropped "
          f"test mission(s) {DROPPED_TEST_MISSIONS} are class {DROPPED_TEST_CLASSES}, so every Class 3/4 test window is kept. "
          f"The control rules out mission composition for Classes 3/4, but it could not have shown anything else.")
elif _mc_zero:
    print("#      -> NO swing at any of the 5 seeds (the control does touch Class 3/4 missions, so this is informative).")
else:
    print("#      -> the mission control MOVES Class 3/4 recall at some seeds: mission composition contributes (see the table).")
print("#      Time-matched control (the same OLD model, scored only from each mission's audio onset onward), paired over 5 seeds:")
for label, k in KEYS[1:]:
    ph, rm = PHASE[k], REMAINDER[k]
    print(f"#        {label}: flight-phase part (time-matched - OLD) {ph['mean']:+.4f} (t p {ph['t_p']:.4f}) | remainder "
          f"(CORRECTED - time-matched) {rm['mean']:+.4f} (t p {rm['t_p']:.4f}) | total {PAIRED[k]['mean']:+.4f}")
for label, k in KEYS[1:]:
    ph, rm, tot = PHASE[k]["mean"], REMAINDER[k]["mean"], PAIRED[k]["mean"]
    if abs(tot) < 0.01:
        print(f"#      -> {label}: mean total swing {tot:+.4f} is too small (< 0.01) to decompose meaningfully.")
        continue
    share = ph / tot
    print(f"#      -> {label}: {100 * share:.0f}% of the mean swing is already present when OLD's model is simply scored on the "
          f"post-onset windows. " + (
              "Most of it is about WHICH PART OF THE FLIGHT is scored, not about correctly aligned audio."
              if share >= 0.5 else
              "Most of it remains after matching the scored flight phase: it comes from training/evaluating with CORRECTED "
              "data (the aligned audio and/or training without the pre-onset windows)."))
print("#  The remainder still combines two things: the audio being aligned correctly, and the model being TRAINED without "
      "the pre-onset windows. Separating them fully needs a third arm that uses OLD alignment trained only on post-onset windows.")
print("#" * 124)


## 8. Final validation
A recap of every check:
- the data is reused exactly (OLD == v18, CORRECTED == v23);
- the seed-42 reuse or reproduction decision;
- per-seed pairing of both alignments;
- the switches and τ invariance;
- the training-data diagnostic;
- the controls.

In [ ]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
trained = {k: r for k, r in run_results.items() if sources[k] == "trained in v24"}
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)
checks = {
    "OLD code == v18 (source sha1)": False not in OLD_SOURCE_CHECK.values(),
    "OLD bins + windows + values + class counts == v18": all(OLD_DATA_CHECK.values()),
    "CORRECTED code == v23 (source sha1)": False not in CORRECTED_SOURCE_CHECK.values(),
    "CORRECTED missions + windows + class counts == v23": all(CORRECTED_DATA_CHECK.values()),
    "exclusion set == the 4 hardcoded missions": EXCLUSION_MATCHES,
    "every trained run starts at its seed's reference weights + RNG": all(r["paired_init"] for r in trained.values()),
    "tau invariance (Condition D ignores timestamps), trained runs": all(r["tau_invariant"] for r in trained.values()),
    "10 slots x own test set present": len(run_results) == len(RUN_ORDER),
    "controls present for all 5 OLD models": all((s, k) in controls for s in SEEDS for k in ("mission", "time")),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
if REUSE_42:
    checks["seed 42 reused: v23 probabilities reproduce v23 exactly"] = True
else:
    checks["seed 42 retrained: reproduces v23 (±0.0005)"] = all(d <= SAME_SEED_TOLERANCE for d, _ in SEED42_REPRO.values())
for name, ok in checks.items():
    print(f"  {name:<64}: {'✅ pass' if ok else '🚨 FAIL'}")
print(f"  {'switches exactly 0 (before, every epoch, best checkpoint)':<64}: ✅ asserted in all {len(trained)} trained runs")
print(f"  {'seed 42':<64}: " + ("REUSED from v23" if REUSE_42 else "RETRAINED (" + "; ".join(REUSE_42_REASON) + ")"))
print(f"  {'Class 4 training windows retained':<64}: {TRAIN_COUNTS['OLD_ALIGNMENT'][4]} -> "
      f"{TRAIN_COUNTS['CORRECTED_ALIGNMENT'][4]} ({TRAIN_RETAINED_PCT[4]:.1f}%; others' mean "
      f"{TRAIN_RETAINED_PCT[_others].mean():.1f}%) -> {'DISPROPORTIONATE' if C4_DISPROPORTIONATE else 'not disproportionate'}")
print(f"  {'mission control touches Class 3/4 test windows':<64}: {'yes' if MISSION_CONTROL_TOUCHES_C34 else 'no (identical by construction)'}")
print(f"  {'confusion matrices printed':<64}: {len(run_results)} runs + {len(controls)} control evaluations")
print("=" * 100)
if not all(checks.values()):
    print("🚨 at least one check FAILED — read the flagged line before using the verdict.")
